# CDI diffractometer with the `cdi_ad_hoc` solver

The workflow from `demo_hkl.ipynb`, using `solver="cdi_ad_hoc"` (`cditools.hkl_solver`).
It returns every motor setting that reaches a reflection. `ad_hoc` returns at most two, and in fixed-incidence modes they can be wrong.

In [1]:
import ad_hoc_diffractometer as ahd
import hklpy2
import hklpy2.user
import numpy as np
from ad_hoc_diffractometer.mode import (
    REQUIRED,
    ConstraintSet,
    DetectorConstraint,
    ReferenceConstraint,
    SampleConstraint,
)
from hklpy2.exceptions import SolverError

## Diffractometer

In [2]:
if "cdi-geometry" not in ahd.list_geometries():
    ahd.register_geometry_file("./diffractometer-geometry.yml", name="cdi-geometry")

diff = hklpy2.creator(name="cdi", geometry="cdi-geometry", solver="cdi_ad_hoc")
diff.beam.energy.put(10.002)  # keV
hklpy2.user.set_diffractometer(diff)
hklpy2.user.add_sample("gold", a=4.08)
diff.core.solver

CDIAdHocSolver(name='cdi_ad_hoc', version='1.0.0', geometry='cdi-geometry')

## Orient the sample

UB from the (1 1 1) and (2 0 0) reflections measured on the beamline.

In [3]:
r1 = hklpy2.user.setor(
    1,
    1,
    1,
    mu=18.2,
    omega1=0,
    chi1=1.4,
    omega2=0,
    chi2=0,
    gamma=0.47,
    delta=31.84,
    name="r111",
)
r2 = hklpy2.user.setor(
    2,
    0,
    0,
    mu=3,
    omega1=0,
    chi1=0,
    omega2=0,
    chi2=0,
    gamma=2.39,
    delta=38.5,
    name="r200",
)
hklpy2.user.calc_UB(r1, r2)

[[0.9356576096693946, 0.9061482661726263, 0.8216015248002685],
 [-0.08919939238156696, 1.0822114170703878, -1.0919939677192774],
 [-1.2199086909439338, 0.6158755368163883, 0.7100063501720328]]

**Check the orientation.** Converting each reflection's angles back to hkl should give (1 1 1) and (2 0 0).

In [4]:
for r in (r1, r2):
    print(r.name, r.pseudos, "->", diff.inverse(diff.RealPosition(**r.reals)))

r111 {'h': 1, 'k': 1, 'l': 1} -> Hklpy2DiffractometerPseudoPos(h=1.0426, k=1.0426, l=1.0426)
r200 {'h': 2, 'k': 0, 'l': 0} -> Hklpy2DiffractometerPseudoPos(h=1.7551, k=0.9068, l=0.9068)


(2 0 0) comes back as about (1.76, 0.91, 0.91). The two reflections are about 19° apart instead of the 54.7° between (1 1 1) and (2 0 0), and their 2θ values (31.8°, 38.6°) don't match gold at 10.002 keV (30.5°, 35.4°). Worth checking the energy, the lattice constant, and the stage signs (the YAML still says `TODO - check +/- signs`). Everything below uses this UB.

## Mode `fixed_gamma_chi2_omega2`

gamma, omega1, omega2 and chi2 stay at 0, and the solver finds mu, chi1 and delta. Each reflection has four settings: two with delta = +2θ and two with delta = −2θ.

In [5]:
diff.core.mode = "fixed_gamma_chi2_omega2"
hklpy2.user.cahkl_table((1, 1, 1), (2, 0, 0), (1, -1, 1))

======== = ========= ====== ========= ====== ==== ===== ========
(hkl)    # mu        omega1 chi1      omega2 chi2 gamma delta   
======== = ========= ====== ========= ====== ==== ===== ========
(1 1 1)  1 17.5321   0      2.1283    0      0    0     30.5103 
(1 1 1)  2 167.0218  0      2.1283    0      0    0     -30.5103
(1 1 1)  3 -17.5321  0      -177.8717 0      0    0     -30.5103
(1 1 1)  4 -167.0218 0      -177.8717 0      0    0     30.5103 
(2 0 0)  1 -34.6996  0      5.4457    0      0    0     35.3746 
(2 0 0)  2 109.9258  0      5.4457    0      0    0     -35.3746
(2 0 0)  3 34.6996   0      -174.5543 0      0    0     -35.3746
(2 0 0)  4 -109.9258 0      -174.5543 0      0    0     35.3746 
(1 -1 1) 1 -9.7095   0      69.3922   0      0    0     30.5103 
(1 -1 1) 2 9.7095    0      -110.6078 0      0    0     -30.5103
(1 -1 1) 3 139.7803  0      69.3922   0      0    0     -30.5103
(1 -1 1) 4 -139.7803 0      -110.6078 0      0    0     30.5103 
======== = ========= ====

Each setting converts back to the hkl it was computed for:

In [6]:
for solution in diff.core.forward((1, -1, 1)):
    print(solution, "->", diff.inverse(solution))

Hklpy2DiffractometerRealPos(mu=-9.7095, omega1=0.0, chi1=69.3922, omega2=0.0, chi2=0.0, gamma=0.0, delta=30.5103) -> Hklpy2DiffractometerPseudoPos(h=1.0, k=-1.0, l=1.0)
Hklpy2DiffractometerRealPos(mu=9.7095, omega1=0.0, chi1=-110.6078, omega2=0.0, chi2=0.0, gamma=0.0, delta=-30.5103) -> Hklpy2DiffractometerPseudoPos(h=1.0, k=-1.0, l=1.0)
Hklpy2DiffractometerRealPos(mu=139.7803, omega1=0.0, chi1=69.3922, omega2=0.0, chi2=0.0, gamma=0.0, delta=-30.5103) -> Hklpy2DiffractometerPseudoPos(h=1.0, k=-1.0, l=1.0)
Hklpy2DiffractometerRealPos(mu=-139.7803, omega1=0.0, chi1=-110.6078, omega2=0.0, chi2=0.0, gamma=0.0, delta=30.5103) -> Hklpy2DiffractometerPseudoPos(h=1.0, k=-1.0, l=1.0)


## Surface normal

The incidence angle is measured from the sample surface, so the solver needs the surface normal `n_hat`, the direction pointing straight out of the surface. In this horizontal scattering geometry the surface faces lab +x (along omega1's axis) with all motors at 0.

hklpy2 wants `n_hat` in crystal (hkl) coordinates, not lab coordinates, so convert it with UB. It isn't simply (1 0 0): that's the crystal's [1 0 0] axis, which points along lab +x only when the crystal's axes line up with the lab's. That was the case in `demo_hkl.ipynb`, where `calc_UB` was commented out and the orientation stayed at the identity.

In [7]:
surface_faces = (1, 0, 0)  # lab direction the surface faces with all motors at 0
n_hat = np.linalg.solve(diff.sample.UB, surface_faces)
n_hat = n_hat / np.abs(n_hat).max()

cos_111 = n_hat @ (1, 1, 1) / (np.linalg.norm(n_hat) * np.sqrt(3))
print(
    "n_hat =",
    np.round(n_hat, 3),
    f"({np.degrees(np.arccos(cos_111)):.1f} deg from (1 1 1))",
)

n_hat = [1.    0.968 0.878] (3.1 deg from (1 1 1))


With this orientation the surface normal is within about 3° of (1 1 1), the direction gold films usually grow along. If you trust that more than the orientation, `n_hat = (1, 1, 1)` is a reasonable alternative.

## Fixed incidence

`fixed_incidence` in `diffractometer-geometry.yml` fixes chi2, omega2, omega1 and gamma (0.47) and also sets the incidence angle. That leaves only mu, chi1 and delta free, but Bragg plus incidence needs four free axes, so in general no setting satisfies all of it. `ad_hoc` returned settings that ignore some of the constraints; `cdi_ad_hoc` explains instead:

In [8]:
diff.core.mode = "fixed_incidence"
diff.core.extras = {"n_hat": tuple(n_hat), "incidence": 3}
try:
    diff.core.forward((1, 1, 1))
except SolverError as exc:
    print(exc)

mode 'fixed_incidence' fixes too many axes: Bragg + incidence needs 4 free axes, but only ['mu', 'chi1', 'delta'] are free.  Remove one of the mode's constraints.


Letting omega1 move (you gave it the same ±12° range as chi1) makes the mode solvable. The next cell adds it for this session only. To keep it, add this to `modes:` in the YAML:

```yaml
  fixed_incidence_gamma:
    constraints:
      - { type: sample, stage: chi2, value: 0.0 }
      - { type: sample, stage: omega2, value: 0.0 }
      - { type: detector, stage: gamma, value: 0.47 }
      - { type: reference, name: incidence, value: 3 }
    computed: [mu, omega1, chi1, delta]
    extras:
      n_hat: REQUIRED
      incidence: REQUIRED
```

In [9]:
diff.core.solver._geom.modes["fixed_incidence_gamma"] = ConstraintSet(
    constraints=[
        SampleConstraint("chi2", 0.0),
        SampleConstraint("omega2", 0.0),
        DetectorConstraint("gamma", 0.47),
        ReferenceConstraint("incidence", 3.0),
    ],
    computed=["mu", "omega1", "chi1", "delta"],
    extras={"n_hat": REQUIRED, "incidence": REQUIRED},
)
diff.core.mode = "fixed_incidence_gamma"
diff.core.extras = {"n_hat": tuple(n_hat), "incidence": 3}
hklpy2.user.cahkl_table((2, 0, 0), (1, -1, 1))

======== = ========= ========= ========= ====== ==== ===== ========
(hkl)    # mu        omega1    chi1      omega2 chi2 gamma delta   
======== = ========= ========= ========= ====== ==== ===== ========
(2 0 0)  1 -59.4245  -36.2947  74.0132   0      0    0.47  35.3719 
(2 0 0)  2 -70.0201  36.9223   -80.7982  0      0    0.47  35.3719 
(2 0 0)  3 59.9055   35.1519   -105.1631 0      0    0.47  -35.3719
(2 0 0)  4 69.6694   -38.1653  99.7771   0      0    0.47  -35.3719
(2 0 0)  5 -110.3306 -141.8347 -80.2229  0      0    0.47  -35.3719
(2 0 0)  6 -120.0945 144.8481  74.8369   0      0    0.47  -35.3719
(2 0 0)  7 109.9799  143.0777  99.2018   0      0    0.47  35.3719 
(2 0 0)  8 120.5755  -143.7053 -105.9868 0      0    0.47  35.3719 
(1 -1 1) 1 -9.7868   -6.5121   71.5833   0      0    0.47  30.507  
(1 -1 1) 2 9.9624    6.2385    -106.8354 0      0    0.47  -30.507 
(1 -1 1) 3 139.4806  -10.4799  73.4666   0      0    0.47  -30.507 
(1 -1 1) 4 -139.2679 9.4412    -105.3042 0      

(1 1 1) has no setting at 3°: its planes are almost parallel to the surface, so it can only be measured near the symmetric setting, where the incidence is close to θ (about 15°).

In [10]:
diff.core.forward((1, 1, 1))

[]

**Check n_hat.** With this normal and omega1 = chi1 = 0, the incidence angle is just mu. These are the incidence angles at which r1 and r2 were measured:

In [11]:
for r in (r1, r2):
    print(r.name, f"incidence = {diff.core.solver.incidence_angle(r.reals):.1f} deg")

r111 incidence = 18.2 deg
r200 incidence = 3.0 deg


Both equal mu, as they should for this mount: r1's (1 1 1) was measured close to its symmetric setting (incidence 18°, θ ≈ 16°), and r2's (2 0 0) at 3°. But the table above only reaches (2 0 0) at 3° with chi1 near ±75–105°, while r2 found it at mu = 3 with no tilt at all. The orientation from r1 and r2 doesn't hold together (see the check near the top), most likely because they were measured on different particles, or the angles need offsets. Redo `calc_UB` with two reflections from the same particle and these settings will change.

## Limits

As in `demo_hkl.ipynb`, limit omega1 and chi1 to ±12°. hklpy2 drops every setting outside the limits.

In [12]:
diff.core.constraints["omega1"].limits = (-12, 12)
diff.core.constraints["chi1"].limits = (-12, 12)

diff.core.mode = "fixed_gamma_chi2_omega2"
hklpy2.user.cahkl_table((1, 1, 1), (2, 0, 0), (1, -1, 1))

======= = ======== ====== ====== ====== ==== ===== ========
(hkl)   # mu       omega1 chi1   omega2 chi2 gamma delta   
======= = ======== ====== ====== ====== ==== ===== ========
(1 1 1) 1 17.5321  0      2.1283 0      0    0     30.5103 
(1 1 1) 2 167.0218 0      2.1283 0      0    0     -30.5103
(2 0 0) 1 -34.6996 0      5.4457 0      0    0     35.3746 
(2 0 0) 2 109.9258 0      5.4457 0      0    0     -35.3746
======= = ======== ====== ====== ====== ==== ===== ========



(1 -1 1) has no rows left: all its settings need chi1 near 69° or −111°. In `fixed_incidence_gamma` nothing fits inside ±12° either; every 3° setting above needs chi1 between about 70° and 110° (either sign):

In [13]:
diff.core.mode = "fixed_incidence_gamma"
diff.core.extras = {"n_hat": tuple(n_hat), "incidence": 3}
diff.core.forward((2, 0, 0))

[]

## Move

`diff.move()` uses the first setting in the list: the least-rotated one inside the limits.

In [14]:
diff.core.mode = "fixed_gamma_chi2_omega2"
diff.move((1, 1, 1))
hklpy2.user.wh()

wavelength=1.2396
pseudos: h=1.0, k=1.0, l=1.0
reals: mu=17.5321, omega1=0, chi1=2.1283, omega2=0, chi2=0, gamma=0, delta=30.5103
